In [2]:
import time
from pathlib import Path
from typing import List, Optional, Literal
import re

import fitz
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from rapidfuzz import fuzz


START_TIME = time.perf_counter()

client = OpenAI()

DATA_DIR = Path("../app/mock_data")
PROTOCOL_PDF = DATA_DIR / "1_Animate_protocol_v2.1 12.06.2018 tracked_18032019_0.pdf"
LAB_MANUAL_PDF = DATA_DIR / "2_ANIMATE lab manual version 2.0 28.03.19_14092023_0.pdf"


def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages = []
    try:
        for i, page in enumerate(doc):
            pages.append({
                "page_number": i + 1,
                "text": page.get_text("text")
            })
    finally:
        doc.close()
    return pages


def build_page_map(pages: list[dict]) -> dict[int, str]:
    return {p["page_number"]: p["text"] for p in pages}


class SampleRef(BaseModel):
    sample_name: str
    sample_type: Optional[str] = None
    source_material: Optional[str] = None
    collection_details: Optional[str] = None
    manual_page: int
    manual_section: Optional[str] = None
    source_text: str


class SampleRefList(BaseModel):
    samples: List[SampleRef]


class ProtocolMatch(BaseModel):
    status: Literal["found", "not_found", "ambiguous"]
    protocol_page: Optional[int] = None
    protocol_section: Optional[str] = None
    evidence: Optional[str] = None
    rationale: Optional[str] = None


def extract_lab_samples(lab_pages: list[dict]) -> pd.DataFrame:
    lab_text = "\n\n".join(
        f"[PAGE {p['page_number']}]\n{p['text']}"
        for p in lab_pages
    )

    prompt = f"""
You are extracting all references to clinical trial samples from a lab manual.

Return every distinct sample that can be collected, processed, shipped, stored, or tested.

For each item, extract:
- sample_name
- sample_type
- source_material
- collection_details
- manual_page
- manual_section
- source_text

Rules:
- manual_page must come from the [PAGE X] markers
- include blood, serum, plasma, urine, saliva, tissue, PK, PD, biomarker, genomic, safety lab, central lab, local lab, and any other sample-related references
- do not invent items
- keep source_text short but sufficient to verify the extraction
- output only valid JSON matching the schema

Lab manual:
{lab_text}
"""

    resp = client.responses.parse(
        model="gpt-4.1",
        input=prompt,
        text_format=SampleRefList,
    )

    sample_refs = resp.output_parsed.samples
    df = pd.DataFrame([s.model_dump() for s in sample_refs])

    if df.empty:
        return pd.DataFrame(columns=[
            "sample_name",
            "sample_type",
            "source_material",
            "collection_details",
            "manual_page",
            "manual_section",
            "source_text",
        ])

    return df[
        [
            "sample_name",
            "sample_type",
            "source_material",
            "collection_details",
            "manual_page",
            "manual_section",
            "source_text",
        ]
    ].copy()


def guess_section(text: str) -> Optional[str]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in lines[:20]:
        if len(line) < 150 and re.search(r"[A-Za-z]", line):
            return line
    return None


def build_protocol_index(protocol_pages: list[dict]) -> list[dict]:
    return [
        {
            "page_number": p["page_number"],
            "section": guess_section(p["text"]),
            "text": p["text"],
        }
        for p in protocol_pages
    ]


def find_protocol_candidates(
    sample_name: str,
    source_text: str = "",
    top_k: int = 5
) -> list[dict]:
    query = f"{sample_name}\n{source_text}".strip().lower()
    results = []

    for page in protocol_index:
        text_lower = page["text"].lower()

        score = max(
            fuzz.partial_ratio(sample_name.lower(), text_lower),
            fuzz.token_set_ratio(sample_name.lower(), text_lower),
            fuzz.partial_ratio(query, text_lower),
        )

        if sample_name.lower() in text_lower:
            score += 20

        results.append({
            "page_number": page["page_number"],
            "section": page["section"],
            "score": score,
            "text": page["text"],
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


def adjudicate_sample_match(sample_row: pd.Series, candidates: list[dict]) -> ProtocolMatch:
    candidate_text = "\n\n".join(
        f"[PROTOCOL PAGE {c['page_number']}]\n[SECTION] {c['section']}\n{c['text'][:4000]}"
        for c in candidates
    )

    prompt = f"""
You are checking whether a sample listed in a lab manual is present in the clinical trial protocol.

Lab manual entry:
- sample_name: {sample_row['sample_name']}
- sample_type: {sample_row.get('sample_type', '')}
- source_material: {sample_row.get('source_material', '')}
- collection_details: {sample_row.get('collection_details', '')}
- manual_page: {sample_row['manual_page']}
- manual_section: {sample_row.get('manual_section', '')}
- source_text: {sample_row['source_text']}

Candidate protocol excerpts:
{candidate_text}

Instructions:
- Decide whether the lab manual sample is clearly present in the protocol.
- Use status:
  - found = clearly present
  - not_found = not present
  - ambiguous = possibly related but not clear enough
- If found or ambiguous, give the best protocol_page and protocol_section.
- evidence should be a short quote/snippet from the protocol excerpts.
- rationale should briefly explain the decision.
- Be conservative. Do not mark found unless the protocol really supports it.
"""

    resp = client.responses.parse(
        model="gpt-4.1",
        input=prompt,
        text_format=ProtocolMatch,
    )
    return resp.output_parsed


def compare_samples_to_protocol(samples_df: pd.DataFrame) -> pd.DataFrame:
    comparison_rows = []

    for _, row in samples_df.iterrows():
        candidates = find_protocol_candidates(
            sample_name=row["sample_name"],
            source_text=row["source_text"],
            top_k=5
        )
        match = adjudicate_sample_match(row, candidates)

        comparison_rows.append({
            "sample_name": row["sample_name"],
            "sample_type": row.get("sample_type"),
            "manual_page": row["manual_page"],
            "manual_section": row.get("manual_section"),
            "protocol_status": match.status,
            "protocol_page": match.protocol_page,
            "protocol_section": match.protocol_section,
            "evidence": match.evidence,
            "rationale": match.rationale,
        })

    return pd.DataFrame(comparison_rows)


def extract_protocol_fragment(
    sample_name: str,
    protocol_page: Optional[int],
    protocol_section: Optional[str] = None,
    fallback_evidence: Optional[str] = None,
    window_before: int = 120,
    window_after: int = 220
) -> Optional[str]:
    if protocol_page is None:
        return None

    text = protocol_page_map.get(protocol_page, "")
    if not text:
        return fallback_evidence

    text_clean = re.sub(r"\s+", " ", text).strip()

    search_terms = [sample_name, protocol_section, fallback_evidence]
    for term in search_terms:
        if not term:
            continue
        term_clean = re.sub(r"\s+", " ", str(term)).strip()
        if not term_clean:
            continue

        match = re.search(re.escape(term_clean), text_clean, flags=re.IGNORECASE)
        if match:
            start = max(0, match.start() - window_before)
            end = min(len(text_clean), match.end() + window_after)
            return text_clean[start:end]

    return fallback_evidence if fallback_evidence else text_clean[:350]


def build_final_table(samples_df: pd.DataFrame, comparison_df: pd.DataFrame) -> pd.DataFrame:
    merged = comparison_df.merge(
        samples_df[["sample_name", "manual_page", "manual_section", "source_text"]],
        on=["sample_name", "manual_page", "manual_section"],
        how="left"
    )

    final_df = merged.rename(columns={
        "sample_name": "sample",
        "manual_page": "lab_manual_page",
        "manual_section": "lab_manual_section",
        "source_text": "lab_manual_text_fragment",
        "protocol_status": "match_status",
        "protocol_page": "trial_protocol_page",
        "protocol_section": "trial_protocol_section",
        "evidence": "trial_protocol_text_fragment",
    })

    final_df["present_in_trial_protocol"] = final_df["match_status"].map({
        "found": "Yes",
        "ambiguous": "Needs review",
        "not_found": "No",
    })

    final_df["flag"] = final_df["match_status"].map({
        "found": "",
        "ambiguous": "REVIEW",
        "not_found": "MISSING FROM TRIAL PROTOCOL",
    })

    final_df["trial_protocol_text_fragment"] = final_df.apply(
        lambda row: extract_protocol_fragment(
            sample_name=row["sample"],
            protocol_page=row["trial_protocol_page"],
            protocol_section=row["trial_protocol_section"],
            fallback_evidence=row["trial_protocol_text_fragment"],
        ),
        axis=1
    )

    final_df = final_df[
        [
            "sample",
            "lab_manual_page",
            "lab_manual_section",
            "lab_manual_text_fragment",
            "present_in_trial_protocol",
            "trial_protocol_page",
            "trial_protocol_section",
            "trial_protocol_text_fragment",
            "flag",
        ]
    ].copy()

    return final_df


def trim_text(value: Optional[str], max_len: int = 250) -> Optional[str]:
    if value is None:
        return None
    value = str(value).strip()
    return value if len(value) <= max_len else value[:max_len] + "..."


protocol_pages = extract_pdf_pages(PROTOCOL_PDF)
lab_pages = extract_pdf_pages(LAB_MANUAL_PDF)

protocol_page_map = build_page_map(protocol_pages)
lab_page_map = build_page_map(lab_pages)

protocol_index = build_protocol_index(protocol_pages)

samples_df = extract_lab_samples(lab_pages)
comparison_df = compare_samples_to_protocol(samples_df)
final_table = build_final_table(samples_df, comparison_df)

display_table = final_table.copy()
display_table["lab_manual_text_fragment"] = display_table["lab_manual_text_fragment"].apply(
    lambda x: trim_text(x, 250)
)
display_table["trial_protocol_text_fragment"] = display_table["trial_protocol_text_fragment"].apply(
    lambda x: trim_text(x, 250)
)

final_table.to_csv("final_sample_link_table.csv", index=False)

print("Saved: final_sample_link_table.csv")
display(display_table)
final_table.to_excel("final_sample_link_table.xlsx", index=False)

TOTAL_SECONDS = time.perf_counter() - START_TIME
print(f"Total runtime: {TOTAL_SECONDS:.2f} seconds")
print(f"Total runtime: {TOTAL_SECONDS/60:.2f} minutes")

Saved: final_sample_link_table.csv


,sample,lab_manual_page,lab_manual_section,lab_manual_text_fragment,present_in_trial_protocol,trial_protocol_page,trial_protocol_section,trial_protocol_text_fragment,flag
0,Formalin-fixed and paraffin-embedded (FFPE) ti...,4,4. Samples required,Formalin-fixed and paraffin-embedded (FFPE) ti...,Yes,51,10.1. Formalin fixed paraffin embedded blocks,he sample tracking website when samples are se...,
1,Peripheral blood in EDTA,4,4. Samples required,Peripheral blood in EDTA,Yes,52,10.2. Peripheral blood samples,blood samples will also be collected as follow...,
2,Peripheral blood in serum gel tube,4,4. Samples required,Peripheral blood in serum gel tube,Yes,52,10.2. Peripheral blood samples,llected as follows: Timepoint Sample required ...,
3,Bone marrow biopsy,3,3. Consent for sample collection,Additional bone marrow biopsy if PET positive ...,Needs review,51,10. EXPLORATORY BIOLOGICAL STUDIES,ANIMATE 48 ANIMATE protocol 2.10 10.04.201812....,REVIEW


Total runtime: 24.65 seconds
Total runtime: 0.41 minutes
